# Import Fama-French datset

In [1]:
import pandas as pd
from tqdm import tqdm
import statsmodels.api as sm

In [ ]:
# pull fama-french dataset

#import pandas_datareader.data as web
#import pandas as pd

# Fetch raw data
#ff3_raw = web.DataReader("F-F_Research_Data_Factors_daily", "famafrench", start="2010-01-01")[0]

# Save raw to file before any transformation
#ff3_raw.to_csv("data/ff3_daily.csv")

#print(f"Saved {ff3_raw.shape[0]} rows to data/ff3_daily.csv")
#print(ff3_raw.head())

/tmp/ipykernel_260675/3873364705.py:7: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff3_raw = web.DataReader("F-F_Research_Data_Factors_daily", "famafrench", start="2010-01-01")[0]


Saved 4085 rows to data/ff3_daily.csv
            Mkt-RF   SMB   HML   RF
Date                               
2010-01-04    1.69  0.61  1.14  0.0
2010-01-05    0.31 -0.64  1.22  0.0
2010-01-06    0.13 -0.23  0.55  0.0
2010-01-07    0.40  0.09  0.96  0.0
2010-01-08    0.33  0.36  0.02  0.0


In [2]:
ff3_raw = pd.read_csv("data/ff3_daily.csv", parse_dates=["Date"], index_col="Date")

# Convert from percentage to decimal
ff3 = ff3_raw / 100
ff3 = ff3.rename(columns={
    "Mkt-RF": "mkt_rf",
    "SMB":    "smb",
    "HML":    "hml",
    "RF":     "rf_ff"
})

print(ff3.shape)
print(ff3.head())

(4085, 4)
            mkt_rf     smb     hml  rf_ff
Date                                     
2010-01-04  0.0169  0.0061  0.0114    0.0
2010-01-05  0.0031 -0.0064  0.0122    0.0
2010-01-06  0.0013 -0.0023  0.0055    0.0
2010-01-07  0.0040  0.0009  0.0096    0.0
2010-01-08  0.0033  0.0036  0.0002    0.0


# Merge with my dataset

In [3]:
# -----------------------------
# 1. Stock returns
# -----------------------------
df = pd.read_csv("data/sp500_data_from_2010.csv", parse_dates=["Date"])

stocks = (
    df[df["Ticker"] != "^GSPC"][["Date", "Ticker", "Adj Close"]]
    .sort_values(["Ticker", "Date"])
)
stocks["stock_return"] = stocks.groupby("Ticker")["Adj Close"].pct_change()
stocks = stocks.dropna(subset=["stock_return"])

# -----------------------------
# 2. Merge with FF3 factors
# -----------------------------
ff3_data = (
    stocks
    .merge(ff3, left_on="Date", right_index=True, how="inner")
)

# Excess return using French's RF
ff3_data["stock_excess"] = ff3_data["stock_return"] - ff3_data["rf_ff"]

print(f"FF3 dataset shape: {ff3_data.shape}")
print(ff3_data[["Date", "Ticker", "stock_excess", "mkt_rf", "smb", "hml"]].head())

FF3 dataset shape: (1727531, 9)
           Date Ticker  stock_excess  mkt_rf     smb     hml
424  2010-01-05      A     -0.010863  0.0031 -0.0064  0.0122
848  2010-01-06      A     -0.003553  0.0013 -0.0023  0.0055
1272 2010-01-07      A     -0.001296  0.0040  0.0009  0.0096
1696 2010-01-08      A     -0.000325  0.0033  0.0036  0.0002
2120 2010-01-11      A      0.000649  0.0013 -0.0013 -0.0026


# Make the 3f regression and compare with CAPM

In [4]:


results_ff3 = []

for ticker, group in tqdm(ff3_data.groupby("Ticker")):
    if len(group) < 100:
        continue
    try:
        X = sm.add_constant(group[["mkt_rf", "smb", "hml"]])
        y = group["stock_excess"]
        model = sm.OLS(y, X).fit()
        results_ff3.append({
            "Ticker":        ticker,
            "alpha":         model.params["const"],
            "beta_mkt":      model.params["mkt_rf"],
            "beta_smb":      model.params["smb"],
            "beta_hml":      model.params["hml"],
            "r2":            model.rsquared,
            "adj_r2":        model.rsquared_adj,
            "p_alpha":       model.pvalues["const"],
            "p_beta_mkt":    model.pvalues["mkt_rf"],
            "p_beta_smb":    model.pvalues["smb"],
            "p_beta_hml":    model.pvalues["hml"],
            "n":             len(group)
        })
    except Exception:
        continue

ff3_full = pd.DataFrame(results_ff3)
print(f"Regressions run: {len(ff3_full)}")
ff3_full.head()

100%|██████████| 423/423 [00:02<00:00, 148.29it/s]

Regressions run: 423


,Ticker,alpha,beta_mkt,beta_smb,beta_hml,r2,adj_r2,p_alpha,p_beta_mkt,p_beta_smb,p_beta_hml,n
0,A,-0.000025,1.108304,0.171275,-0.087396,0.498966,0.498598,0.901429,0.0,4.076684e-07,7.130150e-04,4084
1,AAPL,0.000414,1.116550,-0.265840,-0.403926,0.505183,0.504819,0.034001,0.0,7.787277e-16,1.787381e-56,4084
2,ABT,0.000061,0.755410,-0.329725,-0.139833,0.364035,0.363567,0.721615,0.0,2.871443e-30,1.788034e-10,4084
3,ACGL,0.000293,0.778540,-0.130634,0.575805,0.407577,0.407142,0.103822,0.0,1.701913e-05,7.026526e-127,4084
4,ACN,0.000005,1.001907,-0.088882,-0.047021,0.496183,0.495813,0.978819,0.0,2.557027e-03,3.675760e-02,4084


In [5]:
def summarize_ff3(df, label):
    return {
        "sample":            label,
        "n_stocks":          len(df),
        "median_r2":         df["r2"].median(),
        "median_adj_r2":     df["adj_r2"].median(),
        "mean_r2":           df["r2"].mean(),
        "mean_adj_r2":       df["adj_r2"].mean(),
        "median_beta_mkt":   df["beta_mkt"].median(),
        "median_beta_smb":   df["beta_smb"].median(),
        "median_beta_hml":   df["beta_hml"].median(),
        "median_alpha_ann":  (df["alpha"] * 252).median(),
        "pct_sig_alpha":     (df["p_alpha"] < 0.05).mean() * 100,
        "pct_sig_smb":       (df["p_beta_smb"] < 0.05).mean() * 100,
        "pct_sig_hml":       (df["p_beta_hml"] < 0.05).mean() * 100,
    }

comparison_ff3 = pd.DataFrame([summarize_ff3(ff3_full, "FF3 — Full Sample")])

# Quick R² gain over CAPM
print(f"CAPM median adjusted R²  : 0.337")
print(f"FF3  median adjusted R²  : {ff3_full['adj_r2'].median():.3f}")
print(f"Gain            : {ff3_full['r2'].median() - 0.337:.3f}")
print()
print(comparison_ff3.T)

CAPM median adjusted R²  : 0.337
FF3  median adjusted R²  : 0.373
Gain            : 0.036

                                  0
sample            FF3 — Full Sample
n_stocks                        423
median_r2                  0.373295
median_adj_r2              0.372834
mean_r2                    0.380349
mean_adj_r2                0.379894
median_beta_mkt              0.9413
median_beta_smb            0.062804
median_beta_hml            0.207891
median_alpha_ann            0.03382
pct_sig_alpha              6.382979
pct_sig_smb                81.08747
pct_sig_hml               90.543735


# Test on the extremes

In [9]:
# -----------------------------
# 1. Reload and merge extremes
# -----------------------------
extremes = pd.read_csv("data/sp500_pct_extremes.csv")
extremes["Date"] = pd.to_datetime(extremes["Date"])

ff3_extremes = extremes.merge(
    ff3_data[["Date", "Ticker", "stock_excess", "mkt_rf", "smb", "hml"]],
    on=["Date", "Ticker"],
    how="inner"
)

# Split tails
ff3_pos = ff3_extremes[ff3_extremes["stock_excess"] > 0].copy()
ff3_neg = ff3_extremes[ff3_extremes["stock_excess"] < 0].copy()

print(f"Total extreme obs : {len(ff3_extremes)}")
print(f"Positive tail     : {len(ff3_pos)}")
print(f"Negative tail     : {len(ff3_neg)}")

Total extreme obs : 1705
Positive tail     : 690
Negative tail     : 1015


In [10]:
# -----------------------------
# 2. FF3 tail regression loop
# -----------------------------
def run_ff3_loop(df, label, min_obs=5):
    results = []
    for ticker, group in df.groupby("Ticker"):
        if len(group) < min_obs:
            continue
        try:
            X = sm.add_constant(group[["mkt_rf", "smb", "hml"]])
            y = group["stock_excess"]
            model = sm.OLS(y, X).fit()
            results.append({
                "Ticker":       ticker,
                "alpha":        model.params["const"],
                "beta_mkt":     model.params["mkt_rf"],
                "beta_smb":     model.params["smb"],
                "beta_hml":     model.params["hml"],
                "r2":           model.rsquared,
                "p_alpha":      model.pvalues["const"],
                "p_beta_mkt":   model.pvalues["mkt_rf"],
                "p_beta_smb":   model.pvalues["smb"],
                "p_beta_hml":   model.pvalues["hml"],
                "n":            len(group)
            })
        except Exception:
            continue
    df_out = pd.DataFrame(results)
    df_out["sample"] = label
    return df_out

ff3_tail_pos = run_ff3_loop(ff3_pos, "FF3 — Positive Tail")
ff3_tail_neg = run_ff3_loop(ff3_neg, "FF3 — Negative Tail")

print(f"Positive tail regressions : {len(ff3_tail_pos)}")
print(f"Negative tail regressions : {len(ff3_tail_neg)}")

Positive tail regressions : 42
Negative tail regressions : 61


In [ ]:
# -----------------------------
# 3. Comparison table R2
# -----------------------------
comparison = pd.DataFrame([
    summarize_ff3(ff3_full,     "FF3 — Full Sample"),
    summarize_ff3(ff3_tail_pos, "FF3 — Positive Tail"),
    summarize_ff3(ff3_tail_neg, "FF3 — Negative Tail"),
])

print(comparison[["sample", "median_r2", "median_beta_mkt",
                   "median_beta_smb", "median_beta_hml",
                   "median_alpha_ann", "pct_sig_alpha",
                   "pct_sig_smb", "pct_sig_hml"]].T)

                                  0                    1                    2
sample            FF3 — Full Sample  FF3 — Positive Tail  FF3 — Negative Tail
median_r2                  0.373295             0.520654             0.581733
median_beta_mkt              0.9413             0.216505              0.23334
median_beta_smb            0.062804            -0.311619            -0.006625
median_beta_hml            0.207891            -0.032717            -0.089737
median_alpha_ann            0.03382            49.177992           -42.534714
pct_sig_alpha              6.382979            85.714286            75.409836
pct_sig_smb                81.08747             7.142857             1.639344
pct_sig_hml               90.543735             4.761905             4.918033


## adjusted R2

In [24]:
# -----------------------------
# 1. Updated loop function
# -----------------------------
def run_ff3_loop(df, label, min_obs=5):
    results = []
    for ticker, group in df.groupby("Ticker"):
        if len(group) < min_obs:
            continue
        try:
            X = sm.add_constant(group[["mkt_rf", "smb", "hml"]])
            y = group["stock_excess"]
            model = sm.OLS(y, X).fit()
            results.append({
                "Ticker":       ticker,
                "alpha":        model.params["const"],
                "beta_mkt":     model.params["mkt_rf"],
                "beta_smb":     model.params["smb"],
                "beta_hml":     model.params["hml"],
                "r2":           model.rsquared,
                "adj_r2":       model.rsquared_adj,
                "p_alpha":      model.pvalues["const"],
                "p_beta_mkt":   model.pvalues["mkt_rf"],
                "p_beta_smb":   model.pvalues["smb"],
                "p_beta_hml":   model.pvalues["hml"],
                "n":            len(group)
            })
        except Exception:
            continue
    return pd.DataFrame(results)

# -----------------------------
# 2. Rerun all three samples
# -----------------------------
ff3_full     = run_ff3_loop(ff3_data,   "FF3 — Full Sample")
ff3_tail_pos = run_ff3_loop(ff3_pos,    "FF3 — Positive Tail")
ff3_tail_neg = run_ff3_loop(ff3_neg,    "FF3 — Negative Tail")

print(f"Full sample       : {len(ff3_full)}")
print(f"Positive tail     : {len(ff3_tail_pos)}")
print(f"Negative tail     : {len(ff3_tail_neg)}")

# -----------------------------
# 3. Updated summary function
# -----------------------------
def summarize_ff3(df, label):
    return {
        "sample":           label,
        "n_stocks":         len(df),
        "median_r2":        df["r2"].median(),
        "median_adj_r2":    df["adj_r2"].median(),
        "median_beta_mkt":  df["beta_mkt"].median(),
        "median_beta_smb":  df["beta_smb"].median(),
        "median_beta_hml":  df["beta_hml"].median(),
        "median_alpha_ann": (df["alpha"] * 252).median(),
        "pct_sig_alpha":    (df["p_alpha"] < 0.05).mean() * 100,
        "pct_sig_smb":      (df["p_beta_smb"] < 0.05).mean() * 100,
        "pct_sig_hml":      (df["p_beta_hml"] < 0.05).mean() * 100,
    }

# -----------------------------
# 4. Comparison table
# -----------------------------
comparison = pd.DataFrame([
    summarize_ff3(ff3_full,     "FF3 — Full Sample"),
    summarize_ff3(ff3_tail_pos, "FF3 — Positive Tail"),
    summarize_ff3(ff3_tail_neg, "FF3 — Negative Tail"),
])

print(comparison[["sample", "median_r2", "median_adj_r2", "median_beta_mkt",
                   "median_beta_smb", "median_beta_hml",
                   "median_alpha_ann", "pct_sig_alpha",
                   "pct_sig_smb", "pct_sig_hml"]].T)

Full sample       : 423
Positive tail     : 42
Negative tail     : 61
                                  0                    1                    2
sample            FF3 — Full Sample  FF3 — Positive Tail  FF3 — Negative Tail
median_r2                  0.373295             0.520654             0.581733
median_adj_r2              0.372834             0.072568             0.156952
median_beta_mkt              0.9413             0.216505              0.23334
median_beta_smb            0.062804            -0.311619            -0.006625
median_beta_hml            0.207891            -0.032717            -0.089737
median_alpha_ann            0.03382            49.177992           -42.534714
pct_sig_alpha              6.382979            85.714286            75.409836
pct_sig_smb                81.08747             7.142857             1.639344
pct_sig_hml               90.543735             4.761905             4.918033
